In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import tokenizer
import torch

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
x=pd.read_excel("sentipers.xlsx")
x.head()

,index,sid,text,polarity,file
0,0,rev-1,اینک قصد داریم پرینتر دیگری از پرینترهای لیزری...,0,data/main/HP LaserJet M1132.xml
1,1,rev-2,پرینتری چند کاره از رده‌ی Entry Level یا سطح م...,0,data/main/HP LaserJet M1132.xml
2,2,rev-3,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، ...,0,data/main/HP LaserJet M1132.xml
3,3,rev-4,به صورتی که توانایی کپی کردن، اسکن، فکس، پر...,0,data/main/HP LaserJet M1132.xml
4,4,rev-5,به هر صورت معمولا چیزی که بیشتر کاربران از پری...,2,data/main/HP LaserJet M1132.xml


In [3]:
x=x.drop(columns=["file","sid","index"])
x.head()

,text,polarity
0,اینک قصد داریم پرینتر دیگری از پرینترهای لیزری...,0
1,پرینتری چند کاره از رده‌ی Entry Level یا سطح م...,0
2,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، ...,0
3,به صورتی که توانایی کپی کردن، اسکن، فکس، پر...,0
4,به هر صورت معمولا چیزی که بیشتر کاربران از پری...,2


In [4]:
x.isnull().sum()

text        0
polarity    0
dtype: int64

In [5]:
def convert(p):
    if p in [-2,-1]:
        return 0
    elif p==0:
        return 1
    else:
        return 2




In [6]:
x1=x["text"]
y=x["polarity"].map(convert)
print(y.value_counts())


polarity
2    7959
1    5938
0    1786
Name: count, dtype: int64


In [7]:
tk=AutoTokenizer.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")

tk2=tk(x1.tolist(),
              padding=True,
              truncation=True,
       max_length=126)

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
print(list(tk2.keys()))

['input_ids', 'token_type_ids', 'attention_mask']


In [9]:
print(tk2["input_ids"][:100])

[[2, 9845, 3954, 3194, 30671, 2953, 2036, 43465, 20943, 4913, 21842, 2049, 3131, 2705, 15, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 30671, 1158, 2429, 16283, 2036, 27041, 53063, 15095, 45003, 2177, 2952, 31825, 15, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2031, 2202, 2353, 3113, 2179, 2028, 14890, 2763, 3414, 300, 2046, 3916, 76306, 8008, 9885, 3377, 2043, 3872, 19257, 4781, 2036, 2819, 2081, 2522, 1

In [10]:
from torch.utils.data import Dataset,DataLoader

class SentimentDataset(Dataset):
    def __init__(self,encoding,labels):
        self.encoding = encoding
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key:torch.tensor(val[idx]) for key ,val in self.encoding.items()}
        item['labels']=torch.tensor(self.labels[idx])
        return item

In [11]:
print(x1.shape,y.shape)
print(y.value_counts())

(15683,) (15683,)
polarity
2    7959
1    5938
0    1786
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x1,y,test_size=0.2,random_state=42)

tk_train=tk(x_train.tolist(),padding=True,truncation=True,max_length=126)
tk_test=tk(x_test.tolist(),padding=True,truncation=True,max_length=126)

train_s=SentimentDataset(tk_train,y_train.tolist())
test_s=SentimentDataset(tk_test,y_test.tolist())



In [13]:
train_lo=DataLoader(train_s,batch_size=32,shuffle=True)
test_lo=DataLoader(test_s,batch_size=32,shuffle=True)

print('train batch',len(train_lo))
print('test batch',len(test_lo))

train batch 393
test batch 99


In [14]:
from transformers import AutoModelForSequenceClassification
model=AutoModelForSequenceClassification.from_pretrained(
    "./parsbert",
    num_labels=3

)

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./parsbert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import Trainer,TrainingArguments

ar=TrainingArguments(
    output_dir='./result',
    num_train_epochs=4,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_strategy='epoch',
    evaluation_strategy='epoch',
    logging_steps=30,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,





)
tra=Trainer(
    model=model,
    args=ar,
    train_dataset=train_s,
    eval_dataset=test_s


)
tra.train()